In [1]:
%pip install -q nfl-data-py

import pandas as pd
import numpy as np
import nfl_data_py as nfl


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime

# Set the season to analyze (e.g., 2024). Change if needed.
season = 2025

# Load play-by-play data for the chosen season
pbp = nfl.import_pbp_data(years=[2025])

# Filter for regular season, Week 1
week1 = pbp[(pbp['week'] == 1)]

# Keep only touchdown plays
week1_tds = week1[week1['touchdown'] == 1]

# Count TDs per scorer. Prefer id+name if both available, else fall back to name only
use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week1_tds.columns]
if use_cols:
    scorers = (
        week1_tds.dropna(subset=use_cols)
        .groupby(use_cols)
        .size()
        .reset_index(name='tds')
    )
    if 'td_player_id' in use_cols:
        scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
    else:
        scorers = scorers.rename(columns={'td_player_name': 'player'})
else:
    # Fallback if td_* columns not present; derive from rusher/receiver
    rush = week1_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
    rec = week1_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
    rush.columns = ['player_id', 'player']
    rec.columns = ['player_id', 'player']
    both = pd.concat([rush, rec], ignore_index=True)
    scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

# Show results
scorers.head(50)


2025 done.
Downcasting floats.


,player_id,player,tds
0,00-0030061,Z.Ertz,1
1,00-0030279,K.Allen,1
2,00-0030506,T.Kelce,1
3,00-0030564,D.Hopkins,1
4,00-0032764,D.Henry,2
5,00-0033288,G.Kittle,1
6,00-0033293,A.Jones,1
7,00-0033553,J.Conner,1
8,00-0033858,J.Smith,1
9,00-0033873,P.Mahomes,1


In [3]:
predictions = pd.read_csv('predictions.csv')

#Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge']]

#Sort by predicted_touchdown_probability in descending order
predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

#Show the top 50 players
predictions



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
0,00-0033553,James Conner,RB,ARI,0.604529,-155.0,-0.003314
1,00-0037248,James Cook,RB,BUF,0.585023,105.0,0.097218
2,00-0035700,Josh Jacobs,RB,GB,0.583525,-160.0,-0.031860
3,00-0034844,Saquon Barkley,RB,PHI,0.581412,-185.0,-0.067711
4,00-0038542,Bijan Robinson,RB,ATL,0.575590,-175.0,-0.060773
...,...,...,...,...,...,...,...
409,00-0040584,Gunnar Helm,TE,TEN,0.053807,950.0,-0.041431
410,00-0040170,Elic Ayomanor,WR,TEN,0.053807,600.0,-0.089050
411,00-0039648,David Martin-Robinson,TE,TEN,0.053572,1800.0,0.000940
412,00-0040705,Chimere Dike,WR,TEN,0.053567,2000.0,0.005948


In [4]:
# Join predictions with scorers: prefer player_id, else fall back to name
if 'player_id' in scorers.columns:
    pred_scored = predictions.merge(
        scorers[['player_id', 'tds']], on='player_id', how='inner'
    )
else:
    pred_scored = predictions.merge(
        scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
    )

pred_scored.sort_values(['predicted_touchdown_probability'], ascending=[False])

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,tds
0,00-0033553,James Conner,RB,ARI,0.604529,-155.0,-0.003314,1
1,00-0037248,James Cook,RB,BUF,0.585023,105.0,0.097218,1
2,00-0035700,Josh Jacobs,RB,GB,0.583525,-160.0,-0.031860,1
3,00-0034844,Saquon Barkley,RB,PHI,0.581412,-185.0,-0.067711,1
4,00-0038542,Bijan Robinson,RB,ATL,0.575590,-175.0,-0.060773,1
5,00-0037840,Kyren Williams,RB,LAR,0.573271,-140.0,-0.010063,1
6,00-0032764,Derrick Henry,RB,BAL,0.556274,-145.0,-0.035563,2
7,00-0036158,J.K. Dobbins,RB,DEN,0.529329,160.0,0.144714,1
8,00-0038597,Chase Brown,RB,CIN,0.528726,-150.0,-0.071274,1
9,00-0039361,Bucky Irving,RB,TB,0.527626,-140.0,-0.055708,1


In [5]:
###Simulate betting on the top 10 running backs
def simulate_betting(df, scorers):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True)



In [6]:
# output players from predictions that play for 'PHI', 'KC', 'LAC' or 'DAL'

#find players with model_edge > 0 and price < 500
ev = predictions[predictions['model_edge'] > 0.10] 
ev = ev[ev['price'] < 500]

#sort by model_edge in descending order
ev = ev.sort_values('model_edge', ascending=False)
ev







,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
39,00-0033858,Jonnu Smith,TE,PIT,0.430336,400.0,0.230336
47,00-0038117,Wan'Dale Robinson,WR,NYG,0.414417,400.0,0.214417
36,00-0037256,Rachaad White,RB,TB,0.448418,320.0,0.210323
43,00-0033923,Kareem Hunt,RB,KC,0.422153,370.0,0.209387
56,00-0036894,Pat Freiermuth,TE,PIT,0.399508,425.0,0.209032
54,00-0038544,Quentin Johnston,WR,LAC,0.404211,370.0,0.191445
40,00-0036139,Rico Dowdle,RB,CAR,0.426987,320.0,0.188891
10,00-0036912,DeVonta Smith,WR,PHI,0.540493,180.0,0.183350
15,00-0037744,Trey McBride,TE,ARI,0.516492,200.0,0.183158
11,00-0036158,J.K. Dobbins,RB,DEN,0.529329,160.0,0.144714


In [7]:
simulate_betting(ev, scorers)

{'bets': 23, 'hits': 6, 'hit_rate': 0.261, 'total_profit': 2.5, 'roi': 0.011}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036912,DeVonta Smith,PHI,WR,180.0,0.540493,0.183350,0,False,-10.0,0.0
1,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.529329,0.144714,1,True,16.0,26.0
2,00-0037744,Trey McBride,ARI,TE,200.0,0.516492,0.183158,0,False,-10.0,0.0
3,00-0035676,A.J. Brown,PHI,WR,160.0,0.509543,0.124928,0,False,-10.0,0.0
4,00-0039901,Keon Coleman,BUF,WR,210.0,0.464003,0.141422,1,True,21.0,31.0
5,00-0034827,DJ Moore,CHI,WR,195.0,0.454181,0.115198,0,False,-10.0,0.0
6,00-0037256,Rachaad White,TB,RB,320.0,0.448418,0.210323,0,False,-10.0,0.0
7,00-0034960,Jakobi Meyers,LVR,WR,225.0,0.440407,0.132715,0,False,-10.0,0.0
8,00-0036252,Michael Pittman,IND,WR,215.0,0.440289,0.122829,1,True,21.5,31.5
9,00-0033858,Jonnu Smith,PIT,TE,400.0,0.430336,0.230336,1,True,40.0,50.0


In [ ]:
### Get top 10 rb, wr, te, qb from predictions
top_rb = predictions[predictions['position'] == 'RB'].sort_values('predicted_touchdown_probability', ascending=False).head(10)
top_wr = predictions[predictions['position'] == 'WR'].sort_values('predicted_touchdown_probability', ascending=False).head(15)
top_te = predictions[predictions['position'] == 'TE'].sort_values('predicted_touchdown_probability', ascending=False).head(5)
top_qb = predictions[predictions['position'] == 'QB'].sort_values('predicted_touchdown_probability', ascending=False).head(5)

top_rb



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge
0,00-0033553,James Conner,RB,ARI,0.604529,-155.0,-0.003314
1,00-0037248,James Cook,RB,BUF,0.585023,105.0,0.097218
2,00-0035700,Josh Jacobs,RB,GB,0.583525,-160.0,-0.031860
3,00-0034844,Saquon Barkley,RB,PHI,0.581412,-185.0,-0.067711
4,00-0038542,Bijan Robinson,RB,ATL,0.575590,-175.0,-0.060773
5,00-0036223,Jonathan Taylor,RB,IND,0.573714,-180.0,-0.069143
6,00-0037840,Kyren Williams,RB,LAR,0.573271,-140.0,-0.010063
7,00-0039139,Jahmyr Gibbs,RB,DET,0.571886,-105.0,0.059690
8,00-0032764,Derrick Henry,RB,BAL,0.556274,-145.0,-0.035563
11,00-0036158,J.K. Dobbins,RB,DEN,0.529329,160.0,0.144714


In [9]:
simulate_betting(top_rb, scorers)

{'bets': 10, 'hits': 8, 'hit_rate': 0.8, 'total_profit': 44.36, 'roi': 0.444}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0033553,James Conner,ARI,RB,-155.0,0.604529,-0.003314,1,True,6.451613,16.451613
1,00-0037248,James Cook,BUF,RB,105.0,0.585023,0.097218,1,True,10.500000,20.500000
2,00-0035700,Josh Jacobs,GB,RB,-160.0,0.583525,-0.031860,1,True,6.250000,16.250000
3,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.581412,-0.067711,1,True,5.405405,15.405405
4,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.575590,-0.060773,1,True,5.714286,15.714286
5,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.573714,-0.069143,0,False,-10.000000,0.000000
6,00-0037840,Kyren Williams,LAR,RB,-140.0,0.573271,-0.010063,1,True,7.142857,17.142857
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.571886,0.059690,0,False,-10.000000,0.000000
8,00-0032764,Derrick Henry,BAL,RB,-145.0,0.556274,-0.035563,2,True,6.896552,16.896552
9,00-0036158,J.K. Dobbins,DEN,RB,160.0,0.529329,0.144714,1,True,16.000000,26.000000


In [10]:
#filter top_wr to only include players with model_edge > 0.05 and price < 500
top_wr = top_wr[top_wr['model_edge'] > 0.05]
top_wr = top_wr[top_wr['price'] < 500]
simulate_betting(top_wr, scorers)

{'bets': 11,
 'hits': 3,
 'hit_rate': 0.273,
 'total_profit': -24.5,
 'roi': -0.223}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0036912,DeVonta Smith,PHI,WR,180.0,0.540493,0.183350,0,False,-10.0,0.0
1,00-0039893,Brian Thomas Jr.,JAX,WR,130.0,0.513541,0.078758,1,True,13.0,23.0
2,00-0035676,A.J. Brown,PHI,WR,160.0,0.509543,0.124928,0,False,-10.0,0.0
3,00-0031381,Davante Adams,LAR,WR,145.0,0.496045,0.087882,0,False,-10.0,0.0
4,00-0039337,Malik Nabers,NYG,WR,150.0,0.494154,0.094154,0,False,-10.0,0.0
5,00-0035659,Terry McLaurin,WAS,WR,130.0,0.490237,0.055455,0,False,-10.0,0.0
6,00-0039075,Puka Nacua,LAR,WR,140.0,0.481381,0.064714,0,False,-10.0,0.0
7,00-0039901,Keon Coleman,BUF,WR,210.0,0.464003,0.141422,1,True,21.0,31.0
8,00-0034827,DJ Moore,CHI,WR,195.0,0.454181,0.115198,0,False,-10.0,0.0
9,00-0034960,Jakobi Meyers,LVR,WR,225.0,0.440407,0.132715,0,False,-10.0,0.0


In [11]:
simulate_betting(top_te, scorers)

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 26.5, 'roi': 0.53}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0037744,Trey McBride,ARI,TE,200.0,0.516492,0.183158,0,False,-10.0,0.0
1,00-0033858,Jonnu Smith,PIT,TE,400.0,0.430336,0.230336,1,True,40.0,50.0
2,00-0033885,David Njoku,CLE,TE,230.0,0.422059,0.119029,0,False,-10.0,0.0
3,00-0030506,Travis Kelce,KC,TE,165.0,0.421664,0.044306,1,True,16.5,26.5
4,00-0036894,Pat Freiermuth,PIT,TE,425.0,0.399508,0.209032,0,False,-10.0,0.0


In [12]:
simulate_betting(top_qb, scorers)

{'bets': 5, 'hits': 4, 'hit_rate': 0.8, 'total_profit': 97.83, 'roi': 1.957}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0034857,Josh Allen,BUF,QB,-120.0,0.519603,-0.025852,2,True,8.333333,18.333333
1,00-0039910,Jayden Daniels,WAS,QB,170.0,0.345601,-0.024769,0,False,-10.000000,0.000000
2,00-0039923,J.J. McCarthy,MIN,QB,600.0,0.309832,0.166975,1,True,60.000000,70.000000
3,00-0035710,Daniel Jones,IND,QB,190.0,0.281073,-0.063755,2,True,19.000000,29.000000
4,00-0034796,Lamar Jackson,BAL,QB,205.0,0.276507,-0.051362,1,True,20.500000,30.500000


In [13]:
top_predictors = predictions[predictions['predicted_touchdown_probability'] >= 0.50]
simulate_betting(top_predictors, scorers)

{'bets': 19, 'hits': 12, 'hit_rate': 0.632, 'total_profit': 29.5, 'roi': 0.155}

,player_id,player_display_name,team,position,price,predicted_touchdown_probability,model_edge,tds,hit,profit,return
0,00-0033553,James Conner,ARI,RB,-155.0,0.604529,-0.003314,1,True,6.451613,16.451613
1,00-0037248,James Cook,BUF,RB,105.0,0.585023,0.097218,1,True,10.500000,20.500000
2,00-0035700,Josh Jacobs,GB,RB,-160.0,0.583525,-0.031860,1,True,6.250000,16.250000
3,00-0034844,Saquon Barkley,PHI,RB,-185.0,0.581412,-0.067711,1,True,5.405405,15.405405
4,00-0038542,Bijan Robinson,ATL,RB,-175.0,0.575590,-0.060773,1,True,5.714286,15.714286
5,00-0036223,Jonathan Taylor,IND,RB,-180.0,0.573714,-0.069143,0,False,-10.000000,0.000000
6,00-0037840,Kyren Williams,LAR,RB,-140.0,0.573271,-0.010063,1,True,7.142857,17.142857
7,00-0039139,Jahmyr Gibbs,DET,RB,-105.0,0.571886,0.059690,0,False,-10.000000,0.000000
8,00-0032764,Derrick Henry,BAL,RB,-145.0,0.556274,-0.035563,2,True,6.896552,16.896552
9,00-0036900,Ja'Marr Chase,CIN,WR,-130.0,0.551778,-0.013440,0,False,-10.000000,0.000000


In [14]:
# nfl_teams = pd.read_csv('nfl_teams.csv')
# team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))
# historic_lines = pd.read_csv('spreadspoke_scores.csv')
# week_1_lines = pd.read_csv('week_1_lines.csv')
# nfl_teams = pd.read_csv('nfl_teams.csv')
# team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))

# Only keep the specified columns from historic_lines
# historic_lines = historic_lines[['schedule_season', 'schedule_week', 'team_home', 'team_away',
#                                 'team_favorite_id', 'spread_favorite', 'over_under_line', 'schedule_playoff']]

# historic_lines.tail()


In [15]:
#rename home_team to team_home and away_team to team_away
week_1_lines.rename(columns={'home_team': "team_home", "away_team":"team_away", "over/under": "over_under_line"}, inplace=True)

# Standardize column names and types for merging

# Ensure both DataFrames have the same team column names and types
# For historic_lines, team_home and team_away are already present
# For week_1_lines, ensure team_home and team_away are present and match the format

# Optionally, standardize team names using team_map if needed
if 'team_home' in week_1_lines.columns and 'team_home' in historic_lines.columns:
    # If team names are not already IDs, map them to IDs for consistency
    if week_1_lines['team_home'].iloc[0] in team_map:
        week_1_lines['team_home_id'] = week_1_lines['team_home'].map(team_map)
        week_1_lines['team_away_id'] = week_1_lines['team_away'].map(team_map)
        historic_lines['team_home_id'] = historic_lines['team_home'].map(team_map)
        historic_lines['team_away_id'] = historic_lines['team_away'].map(team_map)
    else:
        # If already IDs, just copy columns for merge
        week_1_lines['team_home_id'] = week_1_lines['team_home']
        week_1_lines['team_away_id'] = week_1_lines['team_away']
        historic_lines['team_home_id'] = historic_lines['team_home']
        historic_lines['team_away_id'] = historic_lines['team_away']


# Determine the favorite team for each row based on the lower (more negative) spread
def get_favorite_id(row):
    # If home spread is less than away spread, home is favorite
    if row['point_1'] < row['point_2']:
        return row['team_home_id']
    else:
        return row['team_away_id']

week_1_lines['team_favorite_id'] = week_1_lines.apply(get_favorite_id, axis=1)

# The spread_favorite column should be the spread value for the favorite team in each row
def get_spread_favorite(row):
    if row['team_favorite_id'] == row['team_home_id']:
        return row['point_1']
    else:
        return row['point_2']

week_1_lines['spread_favorite'] = week_1_lines.apply(get_spread_favorite, axis=1)

week_1_lines['schedule_season'] = 2025
week_1_lines['schedule_week'] = 1
week_1_lines['schedule_playoff'] = False

week_1_lines


NameError: name 'week_1_lines' is not defined

In [ ]:
# Merge week_1_lines with historic_lines, keeping only columns present in both
common_cols = [col for col in week_1_lines.columns if col in historic_lines.columns]
week_1_lines_aligned = week_1_lines[common_cols].copy()
merged_lines = pd.concat([historic_lines, week_1_lines_aligned], ignore_index=True)
merged_lines.tail(20)

# Omit data from seasons before 2020
merged_lines = merged_lines[merged_lines['schedule_season'] >= 2020]

merged_lines.to_csv("historic_lines.csv")



In [ ]:
import nfl_data_py as nfl
import nfl_td_lambda.data_collection as data

nfl_df = data.get_nfl_data([2020,2021,2022,2023,2024])
nfl_2025_df = data.get_nfl_2025_weekly_data()
nfl_df = pd.concat([nfl_df, nfl_2025_df], ignore_index=True)
nfl_df = nfl_df[nfl_df['week'] <= 18]
nfl_df


Downcasting floats.


,player_id,player_display_name,position,recent_team,season,week,carries,rushing_yards,rushing_tds,receptions,...,receiving_tds,opponent_team,wopr,rushing_epa,receiving_epa,target_share,receiving_air_yards,air_yards_share,racr,scored_touchdown
0,00-0019596,Tom Brady,QB,TB,2020,1,3,9.0,1,0,...,0,NO,0.000000,1.505448,0.000000,0.000000,0.0,0.000000,0.000000,1
1,00-0019596,Tom Brady,QB,TB,2020,2,1,0.0,0,0,...,0,CAR,0.000000,-5.488591,0.000000,0.000000,0.0,0.000000,0.000000,0
2,00-0019596,Tom Brady,QB,TB,2020,3,5,0.0,0,0,...,0,DEN,0.000000,-3.811726,0.000000,0.000000,0.0,0.000000,0.000000,0
3,00-0019596,Tom Brady,QB,TB,2020,4,3,-3.0,0,0,...,0,LAC,0.000000,-1.166074,0.000000,0.000000,0.0,0.000000,0.000000,0
4,00-0019596,Tom Brady,QB,TB,2020,5,3,0.0,0,0,...,0,CHI,0.000000,1.146621,0.000000,0.000000,0.0,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26587,00-0040734,TreVeyon Henderson,RB,NE,2025,1,5,27.0,0,6,...,0,LV,0.162323,0.358535,-2.948261,0.133333,-19.0,-0.053824,-1.263158,0
26588,00-0040735,Luther Burden III,WR,CHI,2025,1,0,0.0,0,1,...,0,MIN,0.030810,0.000000,-0.979480,0.029412,-5.0,-0.019011,0.000000,0
26589,00-0040736,Mason Taylor,TE,NYJ,2025,1,0,0.0,0,1,...,0,PIT,0.145985,0.000000,1.461758,0.047619,18.0,0.106509,1.111111,0
26590,00-0040739,Elijah Arroyo,TE,SEA,2025,1,0,0.0,0,1,...,0,SF,0.085948,0.000000,0.328269,0.045455,5.0,0.025381,1.400000,0


In [ ]:
import nfl_td_lambda.data_collection as data
data.get_odds_data([2020,2021,2022,2023,2024,2025], team_map)
data.get_ngs_data_receiving([2024,2025])

,season,week,player_id,avg_cushion,avg_separation,avg_intended_air_yards,percent_share_of_intended_air_yards,catch_percentage,avg_expected_yac,avg_yac_above_expectation
11894,2024,0,00-0034407,8.143947,3.187889,6.720460,13.054215,71.264368,4.465059,1.171554
11895,2024,0,00-0038976,7.996364,5.205588,7.139808,9.146722,75.000000,9.636653,2.777450
11896,2024,0,00-0036849,7.494035,3.368998,11.851452,18.011678,67.741935,3.496919,-0.599777
11897,2024,0,00-0035500,7.475250,3.688801,4.251600,5.757699,74.000000,5.521061,2.513264
11898,2024,0,00-0039868,7.358235,3.805753,13.236604,17.283354,52.830189,5.645541,0.441245
...,...,...,...,...,...,...,...,...,...,...
13470,2025,1,00-0034407,4.136667,1.847583,13.548000,22.009227,60.000000,2.426947,5.316386
13471,2025,1,00-0038041,3.656000,3.438868,5.335000,9.526502,83.333333,1.749837,0.296163
13472,2025,1,00-0031588,3.602000,3.408239,7.817143,15.223681,85.714286,13.942820,-10.319487
13473,2025,1,00-0036407,3.226250,2.784681,11.436250,42.254757,62.500000,1.839441,0.854559


In [ ]:
depth_chart_df = data.get_depth_chart_data([2020, 2021, 2022, 2023, 2024])
depth_chart_df_2025 = data.get_2025_depth_chart_data()

depth_chart_df = pd.concat([depth_chart_df, depth_chart_df_2025], ignore_index=True)
depth_chart_df = depth_chart_df[depth_chart_df['week'] <= 18]

depth_chart_df[depth_chart_df['player_id']=='00-0036912']

,player_id,season,week,depth_chart_rank
14344,00-0036912,2021,1.0,1
14366,00-0036912,2021,2.0,1
14370,00-0036912,2021,3.0,1
14389,00-0036912,2021,4.0,1
14402,00-0036912,2021,5.0,1
...,...,...,...,...
41094,00-0036912,2024,15.0,1
41097,00-0036912,2024,16.0,1
41122,00-0036912,2024,17.0,1
41126,00-0036912,2024,18.0,1


In [ ]:
pbp = nfl.import_pbp_data([2024,2025]).columns
pbp


2024 done.
2025 done.
Downcasting floats.


Index(['play_id', 'game_id', 'old_game_id_x', 'home_team', 'away_team',
       'season_type', 'week', 'posteam', 'posteam_type', 'defteam',
       ...
       'route', 'defense_man_zone_type', 'defense_coverage_type',
       'offense_names', 'defense_names', 'offense_positions',
       'defense_positions', 'offense_numbers', 'defense_numbers',
       'old_game_id'],
      dtype='object', length=398)

In [ ]:
pbp = nfl.import_pbp_data([2024,2025])
pbp[pbp['rusher_player_id'] == '00-0040122']


2024 done.
2025 done.
Downcasting floats.


,play_id,game_id,old_game_id_x,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers,old_game_id
50851,88.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50853,140.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50863,367.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50888,1010.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50889,1032.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50906,1471.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50952,2590.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50954,2640.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50962,2856.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704
50988,3539.0,2025_01_LV_NE,NaN,NE,LV,REG,1,LV,away,NE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025090704


In [ ]:
week_1 = pd.read_csv('data/stats_player_week_2025.csv')
week_1[week_1['player_id']=='00-0040122']

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,team,...,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,fantasy_points,fantasy_points_ppr
953,00-0040122,A.Jeanty,Ashton Jeanty,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,2025,1,REG,LV,...,0,0,NaN,0,0,0,0,0,10.0,12.0


In [6]:
snap_df = nfl.import_snap_counts([2024, 2025])
snap_df.rename(columns={
        'pfr_player_id': 'pfr_id',
    }, inplace=True)
snap_df[snap_df['player']== 'Ashton Jeanty']


/Users/arysuri/Library/Python/3.9/lib/python/site-packages/nfl_data_py/__init__.py:1006: FutureWarning: Behavior when concatenating bool-dtype and numeric-dtype arrays is deprecated; in a future version these will cast to object dtype (instead of coercing bools to numeric values). To retain the old behavior, explicitly cast bool-dtype arrays to numeric dtype.
  df = pandas.concat([pandas.read_parquet(url.format(x)) for x in years])


,game_id,pfr_game_id,season,game_type,week,player,pfr_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct,.progress
800,2025_01_LV_NE,202509070nwe,2025,REG,1,Ashton Jeanty,JeanAs00,RB,LV,NE,54.0,0.86,0.0,0.0,1.0,0.03,NaN


In [62]:
ids = nfl.import_ids()
# filter ids to only include gsis_id and merge_name
ids=ids[['gsis_id', 'pfr_id', 'merge_name']]
ids

,gsis_id,pfr_id,merge_name
0,NaN,NaN,cam ward
1,NaN,NaN,shedeur sanders
2,NaN,NaN,jaxson dart
3,NaN,NaN,jalen milroe
4,NaN,NaN,quinn ewers
...,...,...,...
12110,NaN,NaN,doug brien
12111,NaN,NaN,jeremy brigham
12112,NaN,NaN,vincent brisby
12113,NaN,NaN,bubby brister


In [9]:
import nfl_td_lambda.data_collection as data
import nfl_data_py as nfl
import pandas as pd
data.get_snap_counts([2024,2025]).tail(30)


/Users/arysuri/Library/Python/3.9/lib/python/site-packages/nfl_data_py/__init__.py:1006: FutureWarning: Behavior when concatenating bool-dtype and numeric-dtype arrays is deprecated; in a future version these will cast to object dtype (instead of coercing bools to numeric values). To retain the old behavior, explicitly cast bool-dtype arrays to numeric dtype.
  df = pandas.concat([pandas.read_parquet(url.format(x)) for x in years])


,pfr_id,player_display_name,team,season,week,offense_snap_share,merge_name
1452,ChesJu00,Julius Chestnut,TEN,2025,1,0.11,julius chestnut
1453,OlivBr02,Bryce Oliver,TEN,2025,1,0.05,bryce oliver
1454,MartDa01,David Martin-Robinson,TEN,2025,1,0.02,david martinrobinson
1455,HancBl00,Blake Hance,TEN,2025,1,0.02,blake hance
1456,LeviCo00,Corey Levin,TEN,2025,1,0.02,corey levin
1457,HookAm00,Amani Hooker,TEN,2025,1,0.00,amani hooker
1458,BartCo00,Cody Barton,TEN,2025,1,0.00,cody barton
1459,WoodXa00,Xavier Woods,TEN,2025,1,0.00,xavier woods
1460,BrowJa09,Jarvis Brownlee,TEN,2025,1,0.00,jarvis brownlee
1461,SimmJe01,Jeffery Simmons,TEN,2025,1,0.00,jeffery simmons


In [10]:
week_2_lines = pd.read_csv('data/week_2_lines.csv')
week_2_lines

,game_id,commence_time,in_play,bookmaker,last_update,home_team,away_team,market,label,description,price,point,over/under
0,0c6e5d9821ce0d3e82f3e792879776a6,9/12/2025,False,DraftKings,9/9/2025,Green Bay Packers,Washington Commanders,spreads,Green Bay Packers,NaN,-105,-3.5,48.5
1,0c6e5d9821ce0d3e82f3e792879776a6,9/12/2025,False,DraftKings,9/9/2025,Green Bay Packers,Washington Commanders,spreads,Washington Commanders,NaN,-115,3.5,48.5
2,1ee9ea2c8256bc6be5dd92e60f6c17de,9/14/2025,False,DraftKings,9/9/2025,Detroit Lions,Chicago Bears,spreads,Chicago Bears,NaN,-108,5.5,46.5
3,1ee9ea2c8256bc6be5dd92e60f6c17de,9/14/2025,False,DraftKings,9/9/2025,Detroit Lions,Chicago Bears,spreads,Detroit Lions,NaN,-112,-5.5,46.5
4,ec75e45d75691f810f347e0c5ff25d6e,9/14/2025,False,DraftKings,9/9/2025,Cincinnati Bengals,Jacksonville Jaguars,spreads,Cincinnati Bengals,NaN,-118,-3.0,49.5
5,ec75e45d75691f810f347e0c5ff25d6e,9/14/2025,False,DraftKings,9/9/2025,Cincinnati Bengals,Jacksonville Jaguars,spreads,Jacksonville Jaguars,NaN,-102,3.0,49.5
6,05a5b084f4e19535b2d3f91ef5c00169,9/14/2025,False,DraftKings,9/9/2025,Dallas Cowboys,New York Giants,spreads,Dallas Cowboys,NaN,-110,-5.5,44.5
7,05a5b084f4e19535b2d3f91ef5c00169,9/14/2025,False,DraftKings,9/9/2025,Dallas Cowboys,New York Giants,spreads,New York Giants,NaN,-110,5.5,44.5
8,575c1b169aab675ec72372ccb0f4c55f,9/14/2025,False,DraftKings,9/9/2025,Miami Dolphins,New England Patriots,spreads,Miami Dolphins,NaN,100,-1.5,44.5
9,575c1b169aab675ec72372ccb0f4c55f,9/14/2025,False,DraftKings,9/9/2025,Miami Dolphins,New England Patriots,spreads,New England Patriots,NaN,-120,1.5,44.5


In [11]:
# Transform week_2_lines to the required format
def transform_week2_lines(df, team_map):
    """Transform week_2_lines data to include: team, opponent, spread_line, total_line, implied_total"""
    games = []
    
    for game_id in df['game_id'].unique():
        game_data = df[df['game_id'] == game_id]
        
        # Get home and away teams
        home_team = game_data['home_team'].iloc[0]
        away_team = game_data['away_team'].iloc[0]
        
        # Get the total line (over/under) - should be the same for both teams
        total_line = game_data['over/under'].iloc[0]
        
        # Get spread data for both teams
        home_spread_data = game_data[game_data['label'] == home_team]
        away_spread_data = game_data[game_data['label'] == away_team]
        
        if len(home_spread_data) > 0 and len(away_spread_data) > 0:
            home_spread = home_spread_data['point'].iloc[0]
            away_spread = away_spread_data['point'].iloc[0]
            
            # Map team names to team IDs
            home_team_id = team_map.get(home_team, home_team)
            away_team_id = team_map.get(away_team, away_team)
            
            # Calculate implied totals
            # For home team: implied_total = (total_line / 2) - (spread_line / 2)
            # For away team: implied_total = (total_line / 2) - (spread_line / 2)
            home_implied_total = (total_line / 2) - (home_spread / 2)
            away_implied_total = (total_line / 2) - (away_spread / 2)
            
            # Add home team row
            games.append({
                'team': home_team_id,
                'opponent': away_team_id,
                'spread_line': home_spread,
                'total_line': total_line,
                'implied_total': home_implied_total
            })
            
            # Add away team row
            games.append({
                'team': away_team_id,
                'opponent': home_team_id,
                'spread_line': away_spread,
                'total_line': total_line,
                'implied_total': away_implied_total
            })
    
    # Create final dataframe
    result_df = pd.DataFrame(games)
    
    # Sort by team for consistency
    
    return result_df

# Apply the transformation
week_2_lines_transformed = transform_week2_lines(week_2_lines, team_map)

# Display the result
print("Transformed week_2_lines data:")
print("=" * 50)
print(week_2_lines_transformed)



NameError: name 'team_map' is not defined

In [ ]:
import nfl_td_lambda.data_collection as data
nfl_teams = pd.read_csv('nfl_teams.csv')
team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))
week_2_lines = pd.read_csv('data/week_2_lines.csv')
data.transform_future_odds(week_2_lines, team_map)

,team,opponent,spread_line,total_line,implied_total
0,GB,WAS,-3.5,48.5,26.00
1,WAS,GB,3.5,48.5,22.50
2,DET,CHI,-5.5,46.5,26.00
3,CHI,DET,5.5,46.5,20.50
4,CIN,JAX,-3.0,49.5,26.25
5,JAX,CIN,3.0,49.5,23.25
6,DAL,NYG,-5.5,44.5,25.00
7,NYG,DAL,5.5,44.5,19.50
8,MIA,NE,-1.5,44.5,23.00
9,NE,MIA,1.5,44.5,21.50


In [ ]:
schedule = nfl.import_schedules([2025])
#schedule['home_team'] = schedule['home_team'].replace({'LA': 'LAR', 'LV': 'LVR'})
#schedule['away_team'] = schedule['away_team'].replace({'LA': 'LAR', 'LV': 'LVR'})

week_schedule = schedule[schedule['week'] == 2]

rosters = nfl.import_seasonal_rosters([2025])

#rosters['team'] = rosters['team'].replace({'LA': 'LAR', 'LV': 'LVR'})



opponent_map = {row['home_team']: row['away_team'] for _, row in week_schedule.iterrows()}
opponent_map.update({row['away_team']: row['home_team'] for _, row in week_schedule.iterrows()})

teams_playing = list(opponent_map.keys())
opponent_map



{'GB': 'WAS',
 'BAL': 'CLE',
 'CIN': 'JAX',
 'DAL': 'NYG',
 'DET': 'CHI',
 'MIA': 'NE',
 'NO': 'SF',
 'NYJ': 'BUF',
 'PIT': 'SEA',
 'TEN': 'LA',
 'ARI': 'CAR',
 'IND': 'DEN',
 'KC': 'PHI',
 'MIN': 'ATL',
 'HOU': 'TB',
 'LV': 'LAC',
 'WAS': 'GB',
 'CLE': 'BAL',
 'JAX': 'CIN',
 'NYG': 'DAL',
 'CHI': 'DET',
 'NE': 'MIA',
 'SF': 'NO',
 'BUF': 'NYJ',
 'SEA': 'PIT',
 'LA': 'TEN',
 'CAR': 'ARI',
 'DEN': 'IND',
 'PHI': 'KC',
 'ATL': 'MIN',
 'TB': 'HOU',
 'LAC': 'LV'}

In [14]:
import nfl_data_py as nfl
import pandas as pd
rosters = nfl.import_seasonal_rosters([2025])
sf = rosters[rosters['team']=='SF']

sf[sf['first_name']=='Sincere']


,season,team,position,depth_chart_position,jersey_number,status,player_name,first_name,last_name,birth_date,...,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number,age
1325,2025,SF,RB,RB,35,DEV,Sincere McCormick,Sincere,McCormick,2000-09-10,...,P01,Sincere,MCC594638,55097,32004d43-4359-4638-6f2a-4110341e4bba,2022,2022,None,NaN,24.0


In [27]:
week = nfl.import_weekly_rosters([2025])
#filter to positions RB, WR, TE, QB
week = week[week['position'].isin(['RB', 'WR', 'TE', 'QB'])]
week[week['status']=='ACT']


,season,team,position,depth_chart_position,jersey_number,status,player_name,first_name,last_name,birth_date,...,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number,age
0,2025,PIT,QB,QB,8,ACT,Aaron Rodgers,Aaron,Rodgers,1983-12-02,...,A01,Aaron,ROD339293,29851,3200524f-4433-9293-a3cf-ad7758d03003,2005,2005,GB,24.0,41.766
4,2025,CLE,QB,QB,15,ACT,Joe Flacco,Joseph,Flacco,1985-01-16,...,A01,Joe,FLA009602,33099,3200464c-4100-9602-96e8-665718e215c0,2008,2008,BAL,18.0,40.641
6,2025,WAS,QB,QB,14,ACT,Josh Johnson,Joshua,Johnson,1986-05-15,...,I02,Josh,JOH000001,33241,32004a4f-4800-0001-966a-b4f304503fea,2008,2008,TB,160.0,39.316
7,2025,LA,QB,QB,9,ACT,Matthew Stafford,John,Stafford,1988-02-07,...,A01,Matthew,STA134157,34452,32005354-4113-4157-19b1-37e6835d09b9,2009,2009,DET,1.0,37.582
18,2025,CAR,QB,QB,14,ACT,Andy Dalton,Andrew,Dalton,1987-10-29,...,A01,Andy,DAL659900,37110,32004441-4c65-9900-551d-a31a0423b3ca,2011,2011,CIN,35.0,37.859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,2025,NYJ,TE,TE,85,ACT,Mason Taylor,Mason,Taylor,2004-05-08,...,A01,Mason,TAY566412,58247,32005441-5956-6412-dca1-e3900d5daf1c,2025,2025,NYJ,42.0,21.333
3004,2025,LA,TE,TE,18,ACT,Terrance Ferguson,Terrance,Ferguson,2003-03-07,...,A01,Terrance,FER333882,58251,32004645-5233-3882-9ca7-fa63e767baea,2025,2025,LAR,46.0,22.505
3006,2025,SEA,TE,TE,18,ACT,Elijah Arroyo,Elijah,Arroyo,2003-04-05,...,A01,Elijah,ARR629685,58255,32004152-5262-9685-50ee-580fc8b42629,2025,2025,SEA,50.0,22.426
3010,2025,NO,QB,QB,6,ACT,Tyler Shough,Tyler,Shough,1999-09-28,...,A01,Tyler,SHO768898,58245,32005348-4f76-8898-2122-c8ee34c8b6a6,2025,2025,NO,40.0,25.944
